# Text Summarization

Use AI directly in SQL to summarize text, either one row at a time with a **scalar** function or across many rows with an **aggregate** function.

## What you will learn

- How to register an LLM once and call it from SQL with `aidb.summarize_text`.
- The difference between a **scalar** call (one input row gives one output) and an **aggregate** call (many rows give one result per group).
- How to supply a **custom prompt** with `aidb.summarize_text_config` so the same function can do other jobs, such as scoring sentiment.

## The data

The demo loads a public dataset of movie critic reviews from Hugging Face (`frankier/processed_multiscale_rt_critics`) into a table called `public.movie_reviews`. Each row has a movie title, a critic name, the review text, and the critic's own score.

## Prerequisites

- A database with AIDB installed.
- An Anthropic (Claude) API key.
- Python packages `datasets` and `pandas`, and internet access to download the dataset.

## Claude API Key

This demo registers a Claude Sonnet model that AIDB's SQL functions then use, so an API key is required. Run the next cell and paste your key when prompted. It is held in memory only and passed to the database in a later step.

In [ ]:
# Provide your Claude API Key
import json
from getpass import getpass
claude_api_key = getpass("Claude API key: ")
creds = json.dumps({"api_key": claude_api_key})

## Connect to the database

Creates a SQLAlchemy engine and attaches the `%sql` magics to it. `pool_pre_ping=True` tests connections before reuse so a dropped connection is replaced instead of causing an error. Named parameters are enabled so `%%sql` cells can reference Python variables such as `:creds`.

In [ ]:
from sqlalchemy import create_engine
import os, pwd

PGUSER = pwd.getpwuid(os.geteuid()).pw_name
PGPORT="5432"
PGDATABASE = os.environ.get("PGDATABASE", "aidb")

if PGUSER == "postgres":
    PGSOCKET = "/run/postgresql"
else:
    PGSOCKET = "/tmp"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}&port={PGPORT}"
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## Register Claude Sonnet

`aidb.create_model` registers the Claude Sonnet 5 model inside the database, which allows the AIDB SQL functions in this demo to use it. The model is stored under the name `claude_model_text_summarization`, and every later query refers to that name.

The first statement deletes any earlier registration with the same name so the notebook can be re-run without errors.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'claude_model_text_summarization';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.models WHERE name = v_name) THEN
        PERFORM aidb.delete_model(v_name);
    END IF;

    PERFORM aidb.create_model(
        name        => v_name,
        provider    => 'anthropic_messages',
        config      => aidb.anthropic_messages_config(model => 'claude-sonnet-5'),
        credentials => CAST(:creds AS JSONB),
        replace_credentials => true
    );
END
$do$;

## Sample data

This cell loads critic reviews from the Rotten Tomatoes-based dataset `frankier/processed_multiscale_rt_critics` on Hugging Face. It then prepares the data and writes it to the database:

1. Load the dataset into a pandas DataFrame.
2. Keep only four columns (`movie_title`, `critic_name`, `review_content`, `review_score`) and rename `review_content` to `review`.
3. Drop rows with an empty review.
4. Write the result to `public.movie_reviews`, replacing the table if it exists, in batches of 2,000 rows.

The first run downloads the dataset, so it can take a few minutes depending on your connection.

In [ ]:
from datasets import load_dataset
import pandas as pd

# 1. Load the dataset that includes movie metadata
dataset = load_dataset("frankier/processed_multiscale_rt_critics")
df = dataset["train"].to_pandas()

# 2. Pick and rename the columns you want to keep for AIDB summarization
columns_to_keep = [
    "movie_title",
    "critic_name",
    "review_content",
    "review_score",
]

# Filter down to the desired columns
df_clean = df[
    [col for col in columns_to_keep if col in df.columns]
].copy()
df_clean = df_clean.rename(columns={"review_content": "review"})

# 3. Drop any rows where the review text is empty or null
df_clean = df_clean.dropna(subset=["review"])

# 4. Load into the database
df_clean.to_sql(
    name="movie_reviews",
    con=conn,
    schema="public",
    if_exists="replace",  # 'replace' automatically creates all the new columns
    index=False,
    chunksize=2000,
    method="multi",
)

print(
    f"Successfully imported {len(df_clean):,} reviews with metadata into public.movie_reviews!"
)

## Review data

A quick sanity check on what was loaded: the total number of reviews, the number of distinct movies, and the number of distinct critics.

In [ ]:
%%sql

SELECT COUNT(*) AS reviews_count, COUNT(DISTINCT movie_title) AS movies_count, COUNT(DISTINCT critic_name) AS critics_count
FROM public.movie_reviews;

## Scalar Function

A scalar function produces **one output per input row**. Here `aidb.summarize_text` summarizes a single review.

- `input` is the text to summarize. We prefix the movie title so the model has context.
- `options` names the model to use.
- No prompt is given. Summarization is the built-in behavior, which is why the prompt is "implied".

The query is filtered to one movie and one critic on purpose. A scalar function calls the model once for every row it returns, so a `WHERE` clause or `LIMIT` is the way to keep an experiment fast and inexpensive.

In [ ]:
%%sql

SELECT movie_title, critic_name, review,
    aidb.summarize_text(
    input   => 'Title: ' || movie_title || '; Review: ' ||review,
    options => '{"model": "claude_model_text_summarization"}'
), review_score
FROM public.movie_reviews
WHERE movie_title = 'Star Wars: The Last Jedi'
AND critic_name = 'Jeff Simon';

## Aggregate Function

An aggregate function takes **many rows and returns one result per group**, in the same way that `SUM` or `COUNT` do. `aidb.summarize_text_aggregate` combines all of the reviews in each group into a single summary.

This query groups by `movie_title` for every Star Wars movie, so you get one summary per film that reflects what all of its critics said. We include each critic's name along with the review text, and `reviews_count` shows how many reviews went into each summary. As before, no prompt is provided, so the function summarizes the reviews.

In [ ]:
%%sql

SELECT movie_title,
    COUNT(*) AS reviews_count,
    aidb.summarize_text_aggregate(
    'Title: ' || movie_title || '; critic_name: ' || critic_name || '; Review: ' || review,
    '{"model": "claude_model_text_summarization"}')
FROM public.movie_reviews
WHERE movie_title ILIKE '%Star Wars%'
GROUP BY movie_title;

## Scalar Function with Custom Prompt

`aidb.summarize_text_config` builds the options for the function and lets you supply your own **prompt**, which controls what the output should be. The result is cast to `json` because that is the format `options` expects.

Here the prompt asks for a sentiment score from 0 (negative) to 1 (positive) and nothing else, so each row returns a number instead of a paragraph. You can compare it with the critic's own `review_score` from the earlier query. The same pattern works for tagging topics, extracting a single fact, or translating, simply by changing the prompt.

In [ ]:
%%sql

SELECT movie_title, critic_name, review,
    aidb.summarize_text( input => review,
    options => aidb.summarize_text_config('claude_model_text_summarization', 
      prompt => 'Using a scale of 0 to 1, give a sentiment of the movie review with 0 being negative and 1 as positive. Do not provide any other ouput other than the score.')::json
    ) AS review_score
FROM public.movie_reviews
WHERE movie_title = 'Star Wars: The Last Jedi'
AND critic_name = 'Jeff Simon';

## Aggregate Function with Custom Prompt

This combines both ideas: `aidb.summarize_text_config` supplies a custom prompt, but the call is an **aggregate**. Instead of one score per review, you get a single sentiment score for each Star Wars movie that reflects all of its reviews together, and the results are sorted by that score.

*Note: this may take a little longer to execute as it aggregates many rows.*

In [ ]:
%%sql

SELECT movie_title,
    COUNT(*) AS reviews_count,
    aidb.summarize_text_aggregate(
    review,
    aidb.summarize_text_config('claude_model_text_summarization', 
      prompt => 'Using a scale of 0 to 1, give a sentiment of the movie review with 0 being negative and 1 as positive. Do not provide any other ouput other than the score.')::json) AS review_score
FROM public.movie_reviews
WHERE movie_title ILIKE '%Star Wars%'
GROUP BY movie_title
ORDER BY review_score DESC;

## Recap

You registered a model once and then used it from plain SQL in four ways:

| Call | Rows in | Rows out | Prompt |
|---|---|---|---|
| `aidb.summarize_text` | 1 | 1 | Built-in summary |
| `aidb.summarize_text_aggregate` | many | 1 per group | Built-in summary |
| `aidb.summarize_text` + `summarize_text_config` | 1 | 1 | Custom |
| `aidb.summarize_text_aggregate` + `summarize_text_config` | many | 1 per group | Custom |

Because these are ordinary SQL expressions, they can be combined with `WHERE`, `GROUP BY`, joins, views, and the rest of your existing queries. Keep in mind that each call uses the model, so filter your data first when experimenting.